In [0]:
from pyspark.sql import functions as F

CATALOG = "olist_dev"
S = f"{CATALOG}.silver"
G = f"{CATALOG}.gold"

def write_gold(df, table):
    df = df.withColumn("gold_load_ts", F.current_timestamp())
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{G}.{table}"))
    print(f"{table}: {spark.table(f'{G}.{table}').count()} rows")

In [0]:
orders = spark.table(f"{S}.orders").select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_delivered_customer_date", "order_estimated_delivery_date")

items = spark.table(f"{S}.order_items").select(
    "order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value")

products = spark.table(f"{S}.products").select("product_id", "product_category_name")
trans = spark.table(f"{S}.category_translation").select(
    "product_category_name", "product_category_name_english")

customers = spark.table(f"{S}.customers").select(
    "customer_id", "customer_unique_id", "customer_city", "customer_state")

sellers = spark.table(f"{S}.sellers").select(
    "seller_id", F.col("seller_state").alias("seller_state"))

fact = (items
    .join(orders, "order_id", "inner")
    .join(products, "product_id", "left")
    .join(trans, "product_category_name", "left")
    .join(customers, "customer_id", "left")
    .join(sellers, "seller_id", "left")
    .withColumn("category", F.coalesce("product_category_name_english",
                                       "product_category_name", F.lit("unknown")))
    .withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .withColumn("item_total", F.col("price") + F.col("freight_value"))
    .drop("product_category_name", "product_category_name_english"))

write_gold(fact, "fact_order_items")

fact_order_items: 112650 rows


In [0]:
fact = spark.table(f"{G}.fact_order_items")

valid = fact.filter(~F.col("order_status").isin("canceled", "unavailable"))

monthly = (valid.groupBy("order_month")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.count("*").alias("items_sold"),
         F.round(F.sum("price"), 2).alias("revenue"),
         F.round(F.sum("freight_value"), 2).alias("freight"),
         F.countDistinct("customer_unique_id").alias("unique_customers"))
    .withColumn("avg_order_value", F.round(F.col("revenue") / F.col("total_orders"), 2))
    .orderBy("order_month"))

write_gold(monthly, "gold_monthly_sales")
display(monthly)

gold_monthly_sales: 24 rows


order_month,total_orders,items_sold,revenue,freight,unique_customers,avg_order_value
2016-09,2,5,207.86,71.83,2,103.93
2016-10,290,342,44507.3,6847.22,287,153.47
2016-12,1,1,10.9,8.72,1,10.9
2017-01,787,953,120098.27,16845.19,753,152.6
2017-02,1718,1936,244959.35,38602.34,1693,142.58
2017-03,2617,2975,368341.32,57276.64,2577,140.75
2017-04,2377,2660,353842.98,52005.63,2345,148.86
2017-05,3640,4106,503159.19,79551.64,3569,138.23
2017-06,3205,3571,429916.61,69735.63,3142,134.14
2017-07,3946,4493,492287.3,86466.43,3872,124.76


In [0]:
category = (valid.groupBy("category")
    .agg(F.count("*").alias("units_sold"),
         F.round(F.sum("price"), 2).alias("revenue"),
         F.round(F.avg("price"), 2).alias("avg_price"))
    .orderBy(F.desc("revenue")))
write_gold(category, "gold_category_sales")

top_products = (valid.groupBy("product_id", "category")
    .agg(F.count("*").alias("units_sold"),
         F.round(F.sum("price"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
    .limit(100))
write_gold(top_products, "gold_top_products")

display(category.limit(10))

gold_category_sales: 74 rows
gold_top_products: 100 rows


category,units_sold,revenue,avg_price
health_beauty,9634,1255695.13,130.34
watches_gifts,5970,1198185.21,200.7
bed_bath_table,11097,1035964.06,93.36
sports_leisure,8590,979740.92,114.06
computers_accessories,7781,904322.02,116.22
furniture_decor,8298,727465.05,87.67
housewares,6915,626825.8,90.65
cool_stuff,3779,620770.49,164.27
auto,4204,586585.73,139.53
garden_tools,4328,481009.94,111.14


In [0]:
cust_summary = (valid.groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.round(F.sum("price"), 2).alias("total_spent"),
         F.min("order_purchase_timestamp").alias("first_order_ts"),
         F.max("order_purchase_timestamp").alias("last_order_ts"),
         F.first("customer_state").alias("state"))
    .withColumn("avg_order_value", F.round(F.col("total_spent") / F.col("total_orders"), 2))
    .withColumn("is_repeat_customer", F.col("total_orders") > 1))

write_gold(cust_summary, "gold_customer_summary")

display(cust_summary.agg(
    F.count("*").alias("customers"),
    F.round(100 * F.avg(F.col("is_repeat_customer").cast("int")), 2).alias("repeat_pct")))

gold_customer_summary: 94983 rows


customers,repeat_pct
94983,3.04


In [0]:
reviews = (spark.table(f"{S}.order_reviews")
    .groupBy("order_id")
    .agg(F.round(F.avg("review_score"), 2).alias("avg_review_score")))

o = (spark.table(f"{S}.orders")
    .filter((F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNotNull())
    .join(spark.table(f"{S}.customers").select("customer_id", "customer_state"), "customer_id", "left")
    .join(reviews, "order_id", "left")
    .withColumn("delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("is_late",
        (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")).cast("int"))
    .withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM")))

delivery = (o.groupBy("order_month", "customer_state")
    .agg(F.count("*").alias("delivered_orders"),
         F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
         F.round(100 * F.avg("is_late"), 2).alias("late_pct"),
         F.round(F.avg("avg_review_score"), 2).alias("avg_review_score")))

write_gold(delivery, "gold_delivery_performance")

gold_delivery_performance: 556 rows


In [0]:
%sql
SELECT customer_state, ROUND(AVG(late_pct),2) AS late_pct, ROUND(AVG(avg_review_score),2) AS review
FROM olist_dev.gold.gold_delivery_performance
GROUP BY customer_state ORDER BY late_pct DESC LIMIT 10;

customer_state,late_pct,review
AL,21.12,3.85
MA,16.81,3.96
SE,14.23,3.94
CE,12.84,4.02
RR,12.74,3.85
PI,12.3,4.08
BA,11.93,3.96
RJ,10.43,4.02
PB,10.15,4.15
TO,9.94,4.21


In [0]:
pay = (spark.table(f"{S}.order_payments")
    .groupBy("payment_type")
    .agg(F.count("*").alias("payments"),
         F.round(F.sum("payment_value"), 2).alias("total_value"),
         F.round(F.avg("payment_installments"), 2).alias("avg_installments")))
write_gold(pay, "gold_payment_mix")
display(pay)

gold_payment_mix: 5 rows


payment_type,payments,total_value,avg_installments
credit_card,76795,1.254208419E7,3.51
debit_card,1529,217989.79,1.0
voucher,5775,379436.87,1.0
boleto,19784,2869361.27,1.0
not_defined,3,0.0,1.0


In [0]:
# 1. Fact ka revenue = Silver items ka revenue (sirf valid status wale hi nahi, total)
fact_rev = spark.table(f"{G}.fact_order_items").agg(F.sum("price")).first()[0]
silver_rev = spark.table(f"{S}.order_items").agg(F.sum("price")).first()[0]
print("Fact revenue :", round(fact_rev, 2))
print("Silver revenue:", round(silver_rev, 2))

# 2. Fact mein duplicate rows nahi honi chahiye
dups = (spark.table(f"{G}.fact_order_items")
        .groupBy("order_id", "order_item_id").count().filter("count > 1").count())
print("Duplicate grain rows:", dups)   # 0 aana chahiye

# 3. Monthly revenue ka total <= fact revenue (canceled hata hai)
m_rev = spark.table(f"{G}.gold_monthly_sales").agg(F.sum("revenue")).first()[0]
print("Monthly total:", round(m_rev, 2))

Fact revenue : 13591643.7
Silver revenue: 13591643.7
Duplicate grain rows: 0
Monthly total: 13494400.74


In [0]:
%sql
SHOW TABLES IN olist_dev.gold;

database,tableName,isTemporary
gold,dim_customer_scd2,false
gold,fact_order_items,false
gold,gold_category_sales,false
gold,gold_customer_summary,false
gold,gold_delivery_performance,false
gold,gold_monthly_sales,false
gold,gold_payment_mix,false
gold,gold_top_products,false


In [0]:
dbutils.notebook.exit("gold_ok")